# Quadruped Locomotion with PPO
## A hands-on introduction to deep reinforcement learning

**What you will build:** a neural network that teaches a 4-legged robot to walk, trained entirely from scratch using Proximal Policy Optimization (PPO).

**Concepts covered:**
1. Markov Decision Process — state, action, reward, episode
2. Actor-Critic architecture — two heads, one network
3. Gaussian policy — exploration via learnable noise
4. Generalized Advantage Estimation (GAE) — variance reduction
5. PPO clipped objective — stable policy updates
6. ΔΔθ control — smooth joint motion via acceleration-level actions

**Prerequisites:** basic Python, some familiarity with neural networks (forward pass, loss, backprop).

---
*Simulator: MuJoCo · Framework: PyTorch · Algorithm: PPO*

## Section 1 — Imports & setup

In [ ]:
import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import mujoco
import mediapy as media
import matplotlib.pyplot as plt
from tqdm import trange

os.makedirs("models", exist_ok=True)
os.makedirs("movies", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

## Section 2 — MuJoCo environment

The **robot** has 4 legs, each with 2 joints (hip + knee) → **8 actuated joints** total.

MuJoCo simulates the physics. We drive it by setting `mjdata.ctrl[:]` (joint position targets) every control step.

```
Joint order:  FL-hip  FL-knee  FR-hip  FR-knee  BL-hip  BL-knee  BR-hip  BR-knee
              ch 0    ch 1     ch 2    ch 3     ch 4    ch 5     ch 6    ch 7
```

`SUBSTEPS = 5` means we run 5 physics steps per control step, giving a 10 ms control period at 100 Hz.

In [ ]:
mjmodel = mujoco.MjModel.from_xml_path("dog.xml")
mjdata  = mujoco.MjData(mjmodel)

SUBSTEPS    = 5
CTRL_DT     = mjmodel.opt.timestep * SUBSTEPS   # duration of one control step (s)
EPISODE_DUR = 8.0    # seconds per episode
SETTLE_TIME = 0.3    # seconds to stabilise at reset before training starts

# Standing-pose joint angles (radians)
HIP_INIT  =  0.90
KNEE_INIT = -1.40

# Joint safety limits — the robot cannot exceed these
HIP_LO,  HIP_HI  = HIP_INIT  - 0.5, HIP_INIT  + 0.5
KNEE_LO, KNEE_HI = KNEE_INIT - 0.3, KNEE_INIT + 0.3

# Replicate across all 4 legs
OFFSETS  = np.array([HIP_INIT,  KNEE_INIT]  * 4, dtype=np.float32)
JOINT_LO = np.array([HIP_LO,   KNEE_LO]    * 4, dtype=np.float32)
JOINT_HI = np.array([HIP_HI,   KNEE_HI]    * 4, dtype=np.float32)

STATE_DIM  = 24   # jpos(8) + jvel(8) + delta(8)
ACTION_DIM = 8    # one ΔΔθ per joint
HIDDEN_DIM = 64   # hidden units in actor (small enough for microcontrollers)

print(f"Control rate : {1/CTRL_DT:.0f} Hz  (dt = {CTRL_DT*1000:.1f} ms)")
print(f"Steps/episode: {int(EPISODE_DUR / CTRL_DT)}")
print(f"State dim: {STATE_DIM}   Action dim: {ACTION_DIM}")

## Section 3 — ΔΔθ control & observation

### Why acceleration-level control?

| Level | Action meaning | Problem |
|---|---|---|
| Position | absolute target angle | Discontinuous jumps |
| Velocity | change in angle (Δθ) | Sudden velocity reversals |
| **Acceleration** | **change in velocity (ΔΔθ)** | **Smooth, inertial motion** ✓ |

The network outputs **ΔΔθ** — a small nudge to a persistent **delta buffer**.  
The delta buffer behaves like velocity; the joint target behaves like position.

```
delta_new = clip( delta + action × MAX_DDELTA,  −DELTA_LIMIT, +DELTA_LIMIT )
ctrl_new  = clip( ctrl  + delta_new,             JOINT_LO,     JOINT_HI    )
```

The delta buffer is also **part of the observation** (24 = 8 jpos + 8 jvel + 8 delta), so the policy knows its own momentum and can plan braking vs. acceleration.

In [ ]:
MAX_DDELTA  = 0.02   # how much one action nudges the delta buffer (rad/step)
DELTA_LIMIT = 0.05   # maximum |delta| — caps the effective joint velocity


def apply_action(ctrl, delta, action):
    """ΔΔθ two-level integration: action → delta → ctrl."""
    delta_new = np.clip(delta + action * MAX_DDELTA, -DELTA_LIMIT, DELTA_LIMIT)
    ctrl_new  = np.clip(ctrl  + delta_new,            JOINT_LO,    JOINT_HI)
    return ctrl_new, delta_new


def get_state(delta):
    """Build the 24-dim observation: [jpos | jvel | delta]."""
    jpos = mjdata.qpos[7:15].astype(np.float32)   # 8 joint angles (rad)
    jvel = mjdata.qvel[6:14].astype(np.float32)   # 8 joint velocities (rad/s)
    return np.concatenate([jpos, jvel, delta])


def orientation():
    """Euler angles (roll, pitch, yaw) in radians from the body quaternion."""
    qw, qx, qy, qz = mjdata.qpos[3:7]
    roll  = math.atan2(2*(qw*qx + qy*qz), 1 - 2*(qx**2 + qy**2))
    pitch = math.asin(float(np.clip(2*(qw*qy - qz*qx), -1, 1)))
    yaw   = math.atan2(2*(qw*qz + qx*qy), 1 - 2*(qy**2 + qz**2))
    return roll, pitch, yaw


def is_fallen():
    """True if the robot's body is too low or tilted more than 45°."""
    _, pitch, _ = orientation()
    return mjdata.qpos[2] < 0.03 or abs(pitch) > math.radians(45)


def reset():
    """Reset the simulation to a stable standing pose.
    Returns (ctrl, delta) — both initialised to the neutral pose."""
    neutral = np.array([HIP_INIT, KNEE_INIT] * 4, dtype=np.float32)
    mujoco.mj_resetData(mjmodel, mjdata)
    mjdata.qpos[7:15] = neutral
    mjdata.qpos[2]    = 0.10    # start 10 cm off the ground
    mujoco.mj_forward(mjmodel, mjdata)
    for _ in range(int(SETTLE_TIME / mjmodel.opt.timestep)):
        mjdata.ctrl[:] = neutral
        mujoco.mj_step(mjmodel, mjdata)
    return neutral.copy(), np.zeros(ACTION_DIM, dtype=np.float32)

## Section 4 — Reward function

The reward tells the agent **what we want**. Every weight is a design decision.

| Term | Sign | Meaning |
|---|---|---|
| Velocity tracking | − | penalise deviation from TARGET_VX |
| Alive bonus | + | reward simply staying upright |
| Sideways velocity | − | penalise drifting left/right |
| Action regularisation | − | penalise large accelerations |
| Delta regularisation | − | penalise drifting far from neutral |
| Joint velocity | + | reward active limb movement |
| Joint excursion | + | reward taking big steps |
| Yaw | − | penalise turning (stay straight) |
| Fall | − | one-time large penalty |

**Reward shaping tip:** start with just velocity + alive, then add the regularisers once the robot can walk at all.

In [ ]:
TARGET_VX    = 0.125   # desired forward speed: 1 m in 8 s
VX_TRACK_W   = 4.0     # how hard to penalise speed mismatch
VX_TOL       = 0.02    # dead-zone: no penalty if |vx − target| < VX_TOL

ALIVE_BONUS  = 0.05
SIDE_VEL_W   = 0.10
ACTION_REG_W = 0.0001
DELTA_REG_W  = 0.0002
JVEL_W       = 0.05
JEXC_W       = 0.05
YAW_W        = 0.20
FALL_PENALTY = 5.0


def compute_reward(action, delta, yaw0):
    vx        = float(mjdata.qvel[0])
    vy        = abs(float(mjdata.qvel[1]))
    jvel      = float(np.mean(np.abs(mjdata.qvel[6:14])))
    excursion = float(np.mean(np.abs(mjdata.qpos[7:15] - OFFSETS)))

    _, _, yaw = orientation()
    dyaw = (yaw - yaw0 + math.pi) % (2 * math.pi) - math.pi

    # Velocity tracking: zero at TARGET_VX; dead-zone avoids penalising noise
    vx_err = max(0.0, abs(vx - TARGET_VX) - VX_TOL)

    return (- VX_TRACK_W   * vx_err ** 2
            + ALIVE_BONUS
            - SIDE_VEL_W   * vy
            - ACTION_REG_W * float(np.sum(action ** 2))
            - DELTA_REG_W  * float(np.sum(delta  ** 2))
            + JVEL_W       * jvel
            + JEXC_W       * excursion
            - YAW_W        * dyaw ** 2)

## Section 5 — Actor-Critic network

PPO uses **one network with two output heads**:

```
                    ┌─ Actor  π(a|s)  → mean(8) + log_std(8)  → Gaussian → tanh → action
State(24) → Trunk? ─┤
                    └─ Critic V(s)   → scalar value
```

Here we keep them completely separate (no shared trunk) for simplicity.

### Actor
- 24 → 64 → 8, single hidden layer with ReLU
- Outputs the **mean** of a Gaussian; a learnable `log_std` is separate
- Output squashed with **tanh** → always in (−1, 1)
- Small enough to deploy on an ESP32 (≈ 8 KB of weights)

### Critic
- 24 → 256 → 256 → 1, two hidden layers
- Only used during training to estimate V(s); thrown away at deployment

### Gaussian policy & log-probability
The action is sampled as:  `a = tanh( mean + std × ε )`,  ε ~ N(0,1)

The log-probability (needed for PPO) must correct for the tanh change-of-variables:

$$\log \pi(a|s) = \log \mathcal{N}(\text{raw}|\mu,\sigma) - \sum_i \log(1 - a_i^2)$$

In [ ]:
LOG_STD_INIT = 0.5    # initial exploration level (σ ≈ 1.65)
LOG_STD_MIN  = -2.0   # minimum log-std  (σ ≈ 0.14 — nearly deterministic)
LOG_STD_MAX  =  1.0   # maximum log-std  (σ ≈ 2.72 — very exploratory)


class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()

        # Actor: small, deployable on microcontroller
        self.pi = nn.Sequential(
            nn.Linear(STATE_DIM, HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, ACTION_DIM),
        )
        nn.init.orthogonal_(self.pi[-1].weight, gain=0.01)  # small init → small early actions
        nn.init.zeros_(self.pi[-1].bias)

        # Learnable log-std: one value per action dimension, not tied to the MLP
        self.log_std = nn.Parameter(torch.full((ACTION_DIM,), LOG_STD_INIT))

        # Critic: larger; lives only on the training machine
        self.vf = nn.Sequential(
            nn.Linear(STATE_DIM, 256), nn.ReLU(),
            nn.Linear(256, 256),       nn.ReLU(),
            nn.Linear(256, 1),
        )

    def get_action(self, s, deterministic=False):
        """Sample an action and return (action, log_prob, value).

        deterministic=True  → return the mean (no noise) for evaluation.
        deterministic=False → sample from the Gaussian for exploration.
        """
        mean    = self.pi(s)
        log_std = self.log_std.clamp(LOG_STD_MIN, LOG_STD_MAX)
        std     = log_std.exp()

        raw    = mean if deterministic else mean + std * torch.randn_like(std)
        action = torch.tanh(raw)   # squash to (−1, 1)

        # Log-prob with tanh correction (change-of-variables formula)
        log_prob = (
            -0.5 * ((raw - mean) / (std + 1e-8)) ** 2
            - log_std
            - 0.5 * math.log(2 * math.pi)
            - torch.log(1 - action.pow(2) + 1e-6)
        ).sum(-1)

        value = self.vf(s).squeeze(-1)
        return action, log_prob, value

    def evaluate_actions(self, s, actions_tanh):
        """Re-compute log_prob and entropy for stored actions (used in PPO update)."""
        mean    = self.pi(s)
        log_std = self.log_std.clamp(LOG_STD_MIN, LOG_STD_MAX)
        std     = log_std.exp()
        raw     = torch.atanh(actions_tanh.clamp(-0.999, 0.999))  # invert tanh

        log_prob = (
            -0.5 * ((raw - mean) / (std + 1e-8)) ** 2
            - log_std
            - 0.5 * math.log(2 * math.pi)
            - torch.log(1 - actions_tanh.pow(2) + 1e-6)
        ).sum(-1)

        # Entropy of a Gaussian (used in the exploration bonus term)
        entropy = (log_std + 0.5 * math.log(2 * math.pi * math.e)).sum(-1)
        value   = self.vf(s).squeeze(-1)
        return log_prob, entropy, value


ac  = ActorCritic().to(device)
opt = torch.optim.Adam(ac.parameters(), lr=3e-4, eps=1e-5)

total_params = sum(p.numel() for p in ac.pi.parameters()) + ac.log_std.numel()
print(f"Actor parameters : {total_params:,}  ({total_params*4/1024:.1f} KB)")

## Section 6 — Rollout buffer & Generalized Advantage Estimation

PPO is **on-policy**: collect data with the current policy → update → discard. The buffer holds one batch.

### Generalized Advantage Estimation (GAE)

The **advantage** A(s,a) = Q(s,a) − V(s) measures how much better action *a* was than the critic expected.

GAE computes it as a weighted sum of TD errors:

$$A_t^{\text{GAE}} = \sum_{k=0}^{\infty} (\gamma\lambda)^k \delta_{t+k}, \quad \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

- λ = 0 → pure 1-step TD (low variance, high bias)  
- λ = 1 → full Monte Carlo (high variance, zero bias)  
- λ = 0.95 → good middle ground in practice

In [ ]:
GAMMA              = 0.99    # discount factor
GAE_LAMBDA         = 0.95    # GAE smoothing (0 = TD, 1 = Monte Carlo)
N_STEPS_PER_UPDATE = 4096    # collect this many steps before each update
PPO_EPOCHS         = 10      # passes over the buffer per update
MINIBATCH_SIZE     = 256     # SGD mini-batch size
CLIP_EPS           = 0.2     # PPO clip range
ENTROPY_COEF       = 0.005   # exploration bonus weight
VF_COEF            = 0.5     # critic loss weight
MAX_GRAD           = 0.5     # gradient clip threshold


class RolloutBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.states, self.actions = [], []
        self.log_probs, self.rewards = [], []
        self.values, self.dones = [], []

    def store(self, state, action, log_prob, reward, value, done):
        self.states.append(state);    self.actions.append(action)
        self.log_probs.append(log_prob); self.rewards.append(reward)
        self.values.append(value);    self.dones.append(done)

    def __len__(self):
        return len(self.states)

    def compute_advantages(self, last_value):
        T      = len(self.rewards)
        adv    = np.zeros(T, dtype=np.float32)
        values = np.array(self.values + [last_value], dtype=np.float32)
        rew    = np.array(self.rewards, dtype=np.float32)
        done   = np.array(self.dones,   dtype=np.float32)

        gae = 0.0
        for t in reversed(range(T)):
            delta  = rew[t] + GAMMA * values[t+1] * (1-done[t]) - values[t]
            gae    = delta + GAMMA * GAE_LAMBDA * (1-done[t]) * gae
            adv[t] = gae

        returns = adv + np.array(self.values, dtype=np.float32)
        return returns, adv

    def as_tensors(self, last_value):
        returns, adv = self.compute_advantages(last_value)
        S   = torch.FloatTensor(np.array(self.states)).to(device)
        A   = torch.FloatTensor(np.array(self.actions)).to(device)
        LP  = torch.FloatTensor(np.array(self.log_probs)).to(device)
        R   = torch.FloatTensor(returns).to(device)
        Adv = torch.FloatTensor(adv).to(device)
        Adv = (Adv - Adv.mean()) / (Adv.std() + 1e-8)  # normalise advantages
        return S, A, LP, R, Adv


buf = RolloutBuffer()
print("Buffer ready.")

## Section 7 — PPO update

### The clipped surrogate objective

Let $r_t = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_\text{old}}(a_t|s_t)}$ be the probability ratio. PPO maximises:

$$L^{\text{CLIP}} = \mathbb{E}\bigl[\min\bigl(r_t A_t,\; \text{clip}(r_t, 1{-}\varepsilon, 1{+}\varepsilon)\, A_t\bigr)\bigr]$$

The clip prevents the new policy from moving too far from the old one in a single update — avoiding the catastrophic collapses that plagued earlier policy-gradient methods.

**Total loss** (minimised):
$$L = -L^{\text{CLIP}} + c_v \cdot L^{\text{VF}} - c_e \cdot H[\pi]$$

In [ ]:
def ppo_update(last_value):
    S, A, LP_old, R, Adv = buf.as_tensors(last_value)
    N = S.shape[0]

    for _ in range(PPO_EPOCHS):
        for start in range(0, N, MINIBATCH_SIZE):
            idx = torch.randperm(N, device=device)[start:start + MINIBATCH_SIZE]

            lp_new, entropy, v_new = ac.evaluate_actions(S[idx], A[idx])

            # Probability ratio π_new / π_old  (computed in log-space for stability)
            ratio = (lp_new - LP_old[idx]).exp()

            # Clipped surrogate objective
            surr1   = ratio * Adv[idx]
            surr2   = ratio.clamp(1 - CLIP_EPS, 1 + CLIP_EPS) * Adv[idx]
            pi_loss = -torch.min(surr1, surr2).mean()

            vf_loss = F.mse_loss(v_new, R[idx])

            loss = pi_loss + VF_COEF * vf_loss - ENTROPY_COEF * entropy.mean()

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(ac.parameters(), MAX_GRAD)
            opt.step()

    buf.clear()

print("PPO update function defined.")

## Section 8 — Episode rollout

One function handles both **training** (stochastic, fills the buffer) and **evaluation** (deterministic, optionally renders).

In [ ]:
def run_episode(deterministic=False, render=False):
    """Run one episode.

    Returns (total_reward, distance_m, frames, last_value).
    """
    ctrl, delta = reset()
    x0          = mjdata.qpos[0]
    _, _, yaw0  = orientation()
    total_r     = 0.0
    frames      = []
    renderer    = None

    if render:
        renderer = mujoco.Renderer(mjmodel, height=480, width=640)
        cam = mujoco.MjvCamera()
        mujoco.mjv_defaultFreeCamera(mjmodel, cam)
        cam.distance = 0.8; cam.elevation = -20

    s = get_state(delta)

    for _ in range(int(EPISODE_DUR / CTRL_DT)):
        st = torch.FloatTensor(s).unsqueeze(0).to(device)
        with torch.no_grad():
            action, log_prob, value = ac.get_action(st, deterministic)
        a_np = action.squeeze(0).cpu().numpy()

        ctrl, delta = apply_action(ctrl, delta, a_np)
        mjdata.ctrl[:] = ctrl
        for _ in range(SUBSTEPS):
            mujoco.mj_step(mjmodel, mjdata)

        r    = compute_reward(a_np, delta, yaw0)
        fell = is_fallen()
        if fell:
            r -= FALL_PENALTY
        total_r += r

        if not deterministic:
            buf.store(s, a_np, log_prob.item(), r, value.item(), float(fell))

        s = get_state(delta)

        if render:
            _, _, yaw = orientation()
            cam.lookat[0] = mjdata.qpos[0]
            cam.lookat[1] = mjdata.qpos[1]
            #cam.azimuth   = (math.degrees(yaw) + 180) % 360   # third-person view
            renderer.update_scene(mjdata, cam)
            frames.append(renderer.render().copy())

        if fell:
            break

    if renderer:
        renderer.close()

    dist = mjdata.qpos[0] - x0

    if deterministic or is_fallen():
        last_val = 0.0
    else:
        with torch.no_grad():
            last_val = ac.vf(
                torch.FloatTensor(s).unsqueeze(0).to(device)
            ).squeeze().item()

    return total_r, dist, frames, last_val

print("run_episode() defined.")

## Section 9 — Training loop

**Training configuration** — adjust these before running:

| Parameter | Value | Effect |
|---|---|---|
| `N_EP` | 5000 | More episodes → better policy, longer training |
| `RENDER_EVERY` | 200 | How often to record a video |

On CPU: ~27 minutes for 5000 episodes at ~3 ep/s.

In [ ]:
N_EP         = 5000
RENDER_EVERY = 200

reward_history, dist_history = [], []
best_dist    = -np.inf
best_weights = None
video_count  = 0

for ep in trange(N_EP):
    r, d, _, last_val = run_episode(deterministic=False)
    reward_history.append(r)
    dist_history.append(d)

    if d > best_dist:
        best_dist    = d
        best_weights = {k: v.clone() for k, v in ac.state_dict().items()}

    if len(buf) >= N_STEPS_PER_UPDATE:
        ppo_update(last_val)

    if ep % 10 == 0:
        avg10 = np.mean(dist_history[-10:])
        sigma = math.exp(ac.log_std.data.mean().item())
        print(f"  ep {ep:5d} | dist={d:.3f}m (avg10={avg10:.3f}) | "
              f"reward={r:.1f} | buf={len(buf)} | σ={sigma:.3f} | best={best_dist:.3f}m")

    if ep > 0 and ep % RENDER_EVERY == 0 and best_weights:
        current_w = {k: v.clone() for k, v in ac.state_dict().items()}
        ac.load_state_dict(best_weights)
        _, bd, frames, _ = run_episode(deterministic=True, render=True)
        if frames:
            path = f"movies/rollout_{video_count:03d}_ep{ep:04d}_{bd:.2f}m.mp4"
            media.write_video(path, frames, fps=30)
            print(f"  → Saved {path}")
        video_count += 1
        ac.load_state_dict(current_w)

    if ep % 200 == 199:
        torch.save({"ac": ac.state_dict(), "best_w": best_weights,
                    "best_dist": best_dist, "ep": ep},
                   f"models/checkpoint_ep{ep}.pth")

## Section 10 — Results

In [ ]:
print(f"\n=== Best distance: {best_dist:.3f} m ===\n")

if best_weights:
    torch.save({"ac": best_weights, "best_dist": best_dist}, "models/best.pth")
    print("Saved: models/best.pth")

    # Final evaluation video
    ac.load_state_dict(best_weights)
    _, bd, frames, _ = run_episode(deterministic=True, render=True)
    if frames:
        media.write_video("movies/final.mp4", frames, fps=30)
        media.show_video(frames, fps=30)
        print(f"Saved: movies/final.mp4  (dist = {bd:.3f} m)")

# Learning curves
w      = 20
smooth = lambda x: np.convolve(x, np.ones(w)/w, "valid") if len(x) >= w else x

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(smooth(dist_history),   color="#2ecc71", lw=1.5)
axes[0].set_title("Distance walked per episode (m)", fontsize=12)
axes[0].set_xlabel("Episode"); axes[0].grid(True, alpha=0.3)

axes[1].plot(smooth(reward_history), color="#3498db", lw=1.5)
axes[1].set_title("Total reward per episode", fontsize=12)
axes[1].set_xlabel("Episode"); axes[1].grid(True, alpha=0.3)

plt.suptitle(f"PPO Training — best distance: {best_dist:.3f} m", fontsize=13)
plt.tight_layout()
plt.savefig("models/learning_curves.png", dpi=120)
plt.show()